# Course 11: On-Device AI
## Session 4: Quantization II: training-aware and LLM quantization

**Goal:** train a small CNN with fake quantization and the straight-through estimator and compare it with post-training quantization at 4, 3 and 2 bits, quantize an LLM-like layer's weights in groups and with NF4, tame activation outliers with SmoothQuant-style scaling, count the true bits per weight, and time a 4-bit weight-only kernel.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

### 1. Setup and a float CNN
Session 3 quantized a trained model after training: **post-training quantization** (PTQ). Here we also train *with* quantization in the loop. The model is a small CNN on the 8 × 8 digits (three 3 × 3 convolutions with 16, 32 and 32 channels, the second with stride 2, then a linear layer), trained in float32 first.

Section 6 uses **ONNX Runtime** for a 4-bit kernel; on Colab or Kaggle run `pip install onnx onnxscript onnxruntime` first.

In [1]:
import os
for _v in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ[_v] = "1"                     # one BLAS thread for NumPy; set before NumPy is imported
import time, copy, logging, tempfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.datasets import load_digits

torch.set_num_threads(1)
torch.manual_seed(0); np.random.seed(0)
t_start = time.time()

digits = load_digits()
X8 = torch.tensor(digits.images / 16.0, dtype=torch.float32).unsqueeze(1)     # (1797, 1, 8, 8)
perm = np.random.default_rng(0).permutation(len(X8))
tr, te = perm[:1257], perm[1257:]
Xtr, ytr, Xte, yte = X8[tr], torch.tensor(digits.target[tr]), X8[te], torch.tensor(digits.target[te])

def fq_weight(w, bits):
    # symmetric per-output-channel fake quantization with the straight-through estimator (STE)
    qmax = 2 ** (bits - 1) - 1
    s = w.detach().abs().amax(dim=tuple(range(1, w.dim())), keepdim=True).clamp_min(1e-8) / qmax
    wq = torch.clamp(torch.round(w / s), -qmax, qmax) * s
    return w + (wq - w).detach()          # forward: wq; backward: d(wq)/dw treated as 1

def fq_act(x, t, bits=8):                 # unsigned activations with a fixed (calibrated) range [0, t]
    s = t / (2 ** bits - 1)
    xq = torch.clamp(torch.round(x / s), 0, 2 ** bits - 1) * s
    return x + (xq - x).detach()

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.convs = nn.ModuleList([nn.Conv2d(1, 16, 3, padding=1), nn.Conv2d(16, 32, 3, stride=2, padding=1),
                                    nn.Conv2d(32, 32, 3, padding=1)])
        self.fc = nn.Linear(32 * 4 * 4, 10)
        self.w_bits = None                 # None: float weights; else bits for [conv1, conv2, conv3, fc]
        self.a_max = None                  # None: float activations; else calibrated ranges of the three ReLU outputs

    def forward(self, x, record=None):
        for i, conv in enumerate(self.convs):
            w = conv.weight if self.w_bits is None else fq_weight(conv.weight, self.w_bits[i])
            x = F.relu(F.conv2d(x, w, conv.bias, conv.stride, conv.padding))
            if record is not None:
                record.append(x.detach().max().item())
            if self.a_max is not None:
                x = fq_act(x, self.a_max[i])
        w = self.fc.weight if self.w_bits is None else fq_weight(self.fc.weight, self.w_bits[3])
        return F.linear(x.flatten(1), w, self.fc.bias)

def train(model, epochs, lr, seed=0):
    g = torch.Generator().manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for _ in range(epochs):
        order = torch.randperm(len(Xtr), generator=g)
        for i in range(0, len(Xtr), 64):
            b = order[i:i + 64]
            loss = F.cross_entropy(model(Xtr[b]), ytr[b])
            opt.zero_grad(); loss.backward(); opt.step()
    return model.eval()

def accuracy(model):
    with torch.no_grad():
        return (model(Xte).argmax(1) == yte).float().mean().item()

torch.manual_seed(0)
cnn = train(SmallCNN(), epochs=20, lr=3e-3)
rec = []
with torch.no_grad():
    cnn(Xtr[:200], record=rec)                     # calibrate 8-bit activation ranges on 200 training images
a_max = rec
print(f"float32 CNN: test accuracy {accuracy(cnn):.3f}; "
      f"{sum(p.numel() for p in cnn.parameters()):,d} parameters; activation ranges {np.round(a_max, 2)}")

float32 CNN: test accuracy 0.978; 19,178 parameters; activation ranges [ 1.98  4.92 15.59]


### 2. Post-training quantization against quantization-aware training
**Fake quantization** rounds the weights in the forward pass but keeps float32 copies for the optimizer. Rounding has zero gradient almost everywhere, so training would stop; the **straight-through estimator** (STE; Bengio, Léonard & Courville 2013) pretends that rounding is the identity in the backward pass, so the float weights keep moving. The line `w + (wq - w).detach()` does exactly that: its value is `wq`, its gradient is that of `w`.

We compare, for weights at 4, 3 and 2 bits (per-channel, symmetric) with activations at 8 bits (written W4A8, W3A8, W2A8):

* **PTQ**: quantize the float model as it is;
* **PTQ, first and last layers at 8 bits**: these layers are often the most sensitive, so they are kept at higher precision (mixed precision);
* **PTQ after the same extra epochs**: a control, the float model trained for as many extra epochs as QAT gets, then quantized;
* **QAT**: start from the float model and train 8 more epochs with fake quantization in the loop.

In [2]:
extra = train(copy.deepcopy(cnn), epochs=8, lr=1e-3)          # the control: same extra epochs, in float
rows = []
print(f"{'weights':7s} | PTQ   | PTQ, first and last 8-bit | PTQ after 8 extra float epochs | QAT (8 epochs, STE)")
for bits in [4, 3, 2]:
    res = []
    for base, w_bits in [(cnn, [bits] * 4), (cnn, [8, bits, bits, 8]), (extra, [bits] * 4)]:
        m = copy.deepcopy(base); m.w_bits, m.a_max = w_bits, a_max
        res.append(accuracy(m))
    q = copy.deepcopy(cnn); q.w_bits, q.a_max = [bits] * 4, a_max
    t0 = time.time(); train(q, epochs=8, lr=1e-3); t_qat = time.time() - t0
    res.append(accuracy(q))
    rows.append((bits, *res))
    print(f"W{bits}A8    | {res[0]:.3f} | {res[1]:25.3f} | {res[2]:30.3f} | {res[3]:.3f}  ({t_qat:.1f} s)")
m8 = copy.deepcopy(cnn); m8.w_bits, m8.a_max = [8] * 4, a_max
print(f"W8A8 PTQ for reference: {accuracy(m8):.3f}")

weights | PTQ   | PTQ, first and last 8-bit | PTQ after 8 extra float epochs | QAT (8 epochs, STE)


W4A8    | 0.978 |                     0.974 |                          0.970 | 0.976  (0.8 s)


W3A8    | 0.948 |                     0.976 |                          0.969 | 0.981  (0.6 s)


W2A8    | 0.372 |                     0.963 |                          0.350 | 0.957  (0.6 s)
W8A8 PTQ for reference: 0.978


### 3. LLM weights: group-wise scales and NF4
Language-model decoding is memory-bound (session 1), so LLMs are usually quantized **weight-only**: int4 weights, activations left in 16 bits, and the weights turned back into floats just before the multiply. At 4 bits, one scale per row is too coarse, so the rows are split into **groups** of 32 to 128 consecutive weights, each with its own scale.

We build an LLM-like linear layer (1,024 inputs, 512 outputs) with rows of different scales and a few large weights (0.2% multiplied by 8), and inputs with **outlier channels**: 6 of the 1,024 input features are 30 times larger than the rest, as observed in large Transformers (Dettmers et al. 2022). The measure is the relative error of the layer's *output* on 256 test tokens. All weights and inputs are synthetic.

**NF4** (Dettmers et al. 2023) uses 16 levels placed at quantiles of a normal distribution (with an exact zero) instead of 16 evenly spaced ones; it suits weights that are roughly normal. We build its levels below from `scipy.stats.norm` and compare it with uniform int4 in blocks of 64.

In [3]:
rng = np.random.default_rng(1)
K, N = 1024, 512
W = rng.standard_normal((N, K)) * np.exp(0.5 * rng.standard_normal((N, 1))) / np.sqrt(K)   # rows of different scales
big = rng.random((N, K)) < 0.002
W[big] *= 8                                                                                  # a few large weights
outlier_ch = rng.choice(K, 6, replace=False)
def tokens(n):
    x = rng.standard_normal((n, K)); x[:, outlier_ch] *= 30
    return x
X_cal, X_test = tokens(256), tokens(256)
Y = X_test @ W.T
rel = lambda Yq: np.linalg.norm(Yq - Y) / np.linalg.norm(Y)

def q_sym(x, bits, group=None, axis_rows=True):
    # symmetric quantization; group=None: one scale per row; group=0: one scale for the tensor; else groups along K
    qmax = 2 ** (bits - 1) - 1
    if group == 0:
        s = np.abs(x).max() / qmax
        return np.clip(np.round(x / s), -qmax, qmax) * s
    g = x.shape[1] if group is None else group
    xr = x.reshape(x.shape[0], -1, g)
    s = np.abs(xr).max(axis=2, keepdims=True) / qmax
    return (np.clip(np.round(xr / s), -qmax, qmax) * s).reshape(x.shape)

print("int4 weights, activations unquantized: relative output error")
for name, grp in [("per-tensor", 0), ("per-channel (one scale per row)", None), ("groups of 128", 128),
                  ("groups of 64", 64), ("groups of 32", 32)]:
    print(f"  {name:32s}: {rel(X_test @ q_sym(W, 4, grp).T):.4f}")

off = 0.9677083                                                          # the NF4 construction (Dettmers et al. 2023)
nf4 = np.sort(np.concatenate([norm.ppf(np.linspace(off, 0.5, 9)[:-1]), [0.0], -norm.ppf(np.linspace(off, 0.5, 8)[:-1])]))
nf4 /= np.abs(nf4).max()
print("NF4 levels:", np.round(nf4, 4))

def q_nf4(x, block=64):
    xr = x.reshape(x.shape[0], -1, block)
    absmax = np.abs(xr).max(axis=2, keepdims=True)
    idx = np.abs((xr / absmax)[..., None] - nf4).argmin(axis=-1)            # nearest NF4 level
    return (nf4[idx] * absmax).reshape(x.shape)

for name, Wq in [("uniform int4, blocks of 64", q_sym(W, 4, 64)), ("NF4, blocks of 64", q_nf4(W, 64))]:
    print(f"  {name:32s}: weight error {np.linalg.norm(Wq - W) / np.linalg.norm(W):.4f}, output error {rel(X_test @ Wq.T):.4f}")

int4 weights, activations unquantized: relative output error
  per-tensor                      : 0.6841
  per-channel (one scale per row) : 0.3130
  groups of 128                   : 0.1478
  groups of 64                    : 0.1178
  groups of 32                    : 0.0941
NF4 levels: [-1.     -0.6962 -0.5251 -0.3949 -0.2844 -0.1848 -0.091   0.      0.0796
  0.1609  0.2461  0.3379  0.4407  0.5626  0.723   1.    ]


  uniform int4, blocks of 64      : weight error 0.1454, output error 0.1178
  NF4, blocks of 64               : weight error 0.1065, output error 0.0885


### 4. Activation outliers: SmoothQuant and mixed precision
For **W8A8** (int8 weights *and* activations, which needs int8 matrix-multiply kernels), the outlier channels hurt the activations: one per-tensor activation scale is set by the outliers, so the ordinary channels get only a few levels. Three treatments:

* **baseline**: per-tensor int8 activations, per-channel int8 weights;
* **LLM.int8()-style decomposition** (Dettmers et al. 2022): the few outlier channels (largest calibration value above 6) are multiplied in float, everything else in int8 with one scale per token;
* **SmoothQuant** (Xiao et al. 2023): divide each input channel $j$ by $s_j = \max|X_j|^\alpha / \max|W_j|^{1-\alpha}$ and multiply the matching weight column by $s_j$. The product $XW^\top$ is unchanged, but the difficulty moves from the activations to the weights, which quantize per channel. The factors come from calibration tokens and can be folded into the previous layer, so they cost nothing at run time. We use $\alpha = 0.5$.

In [4]:
def q_tensor(x, bits=8):
    return q_sym(x, bits, 0)

base = q_tensor(X_test) @ q_sym(W, 8).T
outl = np.abs(X_cal).max(0) > 6
Xr, Wr = X_test[:, ~outl], W[:, ~outl]
llm_int8 = X_test[:, outl] @ W[:, outl].T + q_sym(Xr, 8) @ q_sym(Wr, 8).T      # q_sym(Xr, 8): one scale per token (row)
alpha = 0.5
s = np.abs(X_cal).max(0) ** alpha / np.abs(W).max(0) ** (1 - alpha)
smooth = q_tensor(X_test / s) @ q_sym(W * s, 8).T
print(f"W8A8 relative output error: baseline {rel(base):.4f}, LLM.int8()-style ({outl.sum()} float channels) {rel(llm_int8):.4f}, "
      f"SmoothQuant alpha=0.5 {rel(smooth):.4f}")
print(f"largest |activation|, ordinary channels {np.abs(X_cal[:, ~outl]).max():.1f}, outlier channels {np.abs(X_cal[:, outl]).max():.1f}; "
      f"after smoothing: largest channel max / median channel max = "
      f"{np.abs(X_cal / s).max(0).max() / np.median(np.abs(X_cal / s).max(0)):.1f} (before: "
      f"{np.abs(X_cal).max(0).max() / np.median(np.abs(X_cal).max(0)):.1f})")

W8A8 relative output error: baseline 0.0722, LLM.int8()-style (6 float channels) 0.0071, SmoothQuant alpha=0.5 0.0301
largest |activation|, ordinary channels 4.6, outlier channels 108.2; after smoothing: largest channel max / median channel max = 14.9 (before: 36.2)


### 5. Bits per weight are not the model size
A "4-bit" model also stores its scales (and zero points, if any). With a 16-bit scale for every group of $g$ weights, the cost is $b + 16/g$ bits per weight; asymmetric groups add a 16-bit minimum or zero point. llama.cpp's simplest block formats store 32 weights plus one float16 scale (Q4_0 and Q8_0); its "K-quants" use larger blocks with smaller scales. On top of that, models keep some tensors (often the embeddings and the output layer) at higher precision, and at run time the KV cache and activations need RAM as well (session 8).

In [5]:
P = 1e9                                                                       # an example: 1 billion weights
print(f"{'format':52s} | bits per weight | size of 1e9 weights")
for name, bpw in [("float16", 16), ("int8, one float16 scale per row of 4,096", 8 + 16 / 4096),
                  ("llama.cpp Q8_0: blocks of 32, float16 scale", (32 * 8 + 16) / 32),
                  ("int4, one float16 scale per row of 4,096", 4 + 16 / 4096),
                  ("int4, groups of 128, float16 scale", 4 + 16 / 128),
                  ("int4, groups of 64, float16 scale", 4 + 16 / 64),
                  ("llama.cpp Q4_0: blocks of 32, float16 scale", (32 * 4 + 16) / 32),
                  ("int4, groups of 32, float16 scale and float16 zero", 4 + 32 / 32)]:
    print(f"{name:52s} | {bpw:15.3f} | {P * bpw / 8 / 1e6:7.1f} MB")

format                                               | bits per weight | size of 1e9 weights
float16                                              |          16.000 |  2000.0 MB
int8, one float16 scale per row of 4,096             |           8.004 |  1000.5 MB
llama.cpp Q8_0: blocks of 32, float16 scale          |           8.500 |  1062.5 MB
int4, one float16 scale per row of 4,096             |           4.004 |   500.5 MB
int4, groups of 128, float16 scale                   |           4.125 |   515.6 MB
int4, groups of 64, float16 scale                    |           4.250 |   531.2 MB
llama.cpp Q4_0: blocks of 32, float16 scale          |           4.500 |   562.5 MB
int4, groups of 32, float16 scale and float16 zero   |           5.000 |   625.0 MB


### 6. Does weight-only int4 make a memory-bound layer faster?
Fewer bytes per weight should speed up a memory-bound matrix–vector product, but only if the kernel can unpack the 4-bit weights quickly. We build a 4,096 × 4,096 float32 `MatMul` as an ONNX model and quantize its weights with ONNX Runtime's `MatMulNBitsQuantizer` (symmetric int4, blocks of 32 or 128). With `accuracy_level=4`, the kernel also quantizes the activations to int8 on the fly and uses integer dot products. For comparison, PyTorch runs the same product with bfloat16 weights (2 bytes each). Batch 1 is one token being generated; 16 tokens is a short prompt.

In [6]:
import onnx
from onnx import helper, numpy_helper, TensorProto
import onnxruntime as ort
from onnxruntime.quantization.matmul_nbits_quantizer import MatMulNBitsQuantizer
logging.getLogger("onnxruntime").setLevel(logging.ERROR)

Kb = Nb = 4096
Wbig = (np.random.default_rng(2).standard_normal((Kb, Nb)) / np.sqrt(Kb)).astype(np.float32)
graph = helper.make_graph([helper.make_node("MatMul", ["X", "W"], ["Y"])], "matmul",
                          [helper.make_tensor_value_info("X", TensorProto.FLOAT, ["M", Kb])],
                          [helper.make_tensor_value_info("Y", TensorProto.FLOAT, ["M", Nb])],
                          [numpy_helper.from_array(Wbig, "W")])
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
model.ir_version = 10
tmp = tempfile.mkdtemp()
paths = {"float32": os.path.join(tmp, "fp32.onnx")}
onnx.save(model, paths["float32"])
for block in [32, 128]:
    q = MatMulNBitsQuantizer(onnx.load(paths["float32"]), block_size=block, is_symmetric=True, accuracy_level=4)
    q.process()
    paths[f"int4, blocks of {block}"] = os.path.join(tmp, f"int4_{block}.onnx")
    onnx.save(q.model.model, paths[f"int4, blocks of {block}"])
opts = ort.SessionOptions(); opts.intra_op_num_threads, opts.inter_op_num_threads = 1, 1
sess = {k: ort.InferenceSession(p, opts, providers=["CPUExecutionProvider"]) for k, p in paths.items()}

def compare_ms(fns, reps=60, warmup=5):
    for f in fns.values():
        for _ in range(warmup):
            f()
    ts = {k: [] for k in fns}
    for _ in range(reps):
        for k, f in fns.items():
            t0 = time.perf_counter(); f(); ts[k].append(time.perf_counter() - t0)
    return {k: float(np.median(v)) * 1e3 for k, v in ts.items()}

Wt32 = torch.tensor(Wbig.T.copy()); Wt16 = Wt32.to(torch.bfloat16)
for M in [1, 16]:
    xm = np.random.default_rng(3).standard_normal((M, Kb)).astype(np.float32)
    xt32 = torch.tensor(xm); xt16 = xt32.to(torch.bfloat16)
    y32 = sess["float32"].run(None, {"X": xm})[0]
    fns = {k: (lambda s=s: s.run(None, {"X": xm})) for k, s in sess.items()}
    fns["PyTorch float32"] = lambda: F.linear(xt32, Wt32)
    fns["PyTorch bfloat16 weights"] = lambda: F.linear(xt16, Wt16)
    with torch.inference_mode():
        t = compare_ms(fns)
    print(f"\n{M} token(s): {'runtime and weights':28s} | file or weight bytes | time (ms) | speed-up | output error")
    for k in fns:
        if k in sess:
            nbytes = os.path.getsize(paths[k]); err = np.linalg.norm(sess[k].run(None, {"X": xm})[0] - y32) / np.linalg.norm(y32)
        else:
            nbytes = (Wt32 if "32" in k else Wt16).numel() * (4 if "32" in k else 2)
            with torch.inference_mode():
                out = F.linear(xt32, Wt32) if "32" in k else F.linear(xt16, Wt16).float()
            err = np.linalg.norm(out.numpy() - y32) / np.linalg.norm(y32)
        base_t = t["float32"] if k in sess else t["PyTorch float32"]
        print(f"{'':10s}{k:28s} | {nbytes:19,d} | {t[k]:9.3f} | {base_t / t[k]:8.2f} | {err:.4f}")


1 token(s): runtime and weights          | file or weight bytes | time (ms) | speed-up | output error
          float32                      |          67,108,971 |     4.727 |     1.00 | 0.0000
          int4, blocks of 32           |          10,486,029 |     1.462 |     3.23 | 0.0848
          int4, blocks of 128          |           8,913,163 |     1.171 |     4.04 | 0.1015
          PyTorch float32              |          67,108,864 |     4.796 |     1.00 | 0.0000
          PyTorch bfloat16 weights     |          33,554,432 |     2.455 |     1.95 | 0.0029



16 token(s): runtime and weights          | file or weight bytes | time (ms) | speed-up | output error
          float32                      |          67,108,971 |     9.515 |     1.00 | 0.0000
          int4, blocks of 32           |          10,486,029 |     6.058 |     1.57 | 0.0857
          int4, blocks of 128          |           8,913,163 |     4.161 |     2.29 | 0.1032
          PyTorch float32              |          67,108,864 |    11.573 |     1.00 | 0.0000
          PyTorch bfloat16 weights     |          33,554,432 |     4.127 |     2.80 | 0.0029


### 7. Exercises

In [7]:
# TODO: SmoothQuant's alpha. Repeat section 4 for alpha in (0, 0.25, 0.5, 0.75, 1) and report the W8A8 output error.
# alpha = 0 leaves the activations alone; alpha = 1 moves all the difficulty into the weights. Where is the best alpha
# here, and why do both ends do worse?
def alpha_sweep():
    return None          # replace with a list of (alpha, relative error)

r = alpha_sweep()
print("not done yet" if r is None else r)

not done yet


In [8]:
# TODO: activation-aware scaling in the spirit of AWQ (Lin et al. 2024). Before int4 quantization with groups of 128,
# multiply the weight columns of the outlier channels by a factor c (and divide those input channels by c), so that
# the weights that meet large activations get finer steps. Try c in (1, 2, 4) and report the output error.
def awq_style(c):
    return None          # replace with the relative output error

r = [awq_style(c) for c in (1, 2, 4)]
print("not done yet" if r[0] is None else r)
print(f"runtime so far {time.time() - t_start:.0f} s")

not done yet
runtime so far 11 s


### What to notice
(Written from the saved run. Accuracies and errors are the same on every run. The timings in section 6 vary: the saved run was made while another program used three of this machine's four cores, and in a quieter test run the 4-bit kernel was faster still.)

1. **PTQ is enough at 4 bits; below that, something must change.** The float CNN scored 0.978, and W8A8 PTQ kept 0.978. At 4-bit weights every variant stayed within 0.008 of float (0.970–0.978). At 3 bits PTQ lost 3 points (0.948) and QAT recovered them (0.981). At 2 bits PTQ collapsed to 0.372 and QAT reached 0.957. The control shows that the gain came from training *with* quantization: 8 extra float epochs followed by PTQ gave 0.350.
2. **Against expectation, mixed precision matched QAT at 2 bits.** Keeping only the first and last layers at 8 bits rescued 2-bit PTQ to 0.963, with no training at all. QAT cost under a second per run here, but on a real model it costs a sizeable fraction of the original training. These are single runs on 540 test images (one image is 0.0019), so differences below about 0.01 are noise.
3. **Groups shrink the error.** For int4 weights of the LLM-like layer, the relative output error was 0.684 with one scale per tensor, 0.313 with one per row, and 0.148, 0.118 and 0.094 with groups of 128, 64 and 32 weights. **NF4** beat uniform int4 with the same blocks of 64: weight error 0.1065 against 0.1454, output error 0.0885 against 0.1178, for the same 4 bits.
4. **Activation outliers ruin per-tensor int8.** Ordinary input channels stayed below 4.6 while the six outlier channels reached 108.2. W8A8 with one activation scale gave an output error of 0.0722. SmoothQuant ($\alpha = 0.5$) cut it to 0.0301 by narrowing the spread of channel ranges (largest over median channel maximum: 36.2 before, 14.9 after). The LLM.int8()-style split cut it to 0.0071, at the price of a second, floating-point product for the six outlier channels.
5. **Bits per weight are not the size.** With one float16 scale per block of 32, llama.cpp's Q4_0 costs 4.5 bits per weight: 562.5 MB for a billion weights, not 500 MB. Groups of 32 with a float16 scale and zero point cost 5.0 bits (625 MB).
6. **Fewer bytes made the memory-bound product faster, but by less than the byte ratio.** At one token the float32 product read 67 MB and took 4.727 ms. The int4 versions were 6.4 and 7.5 times smaller and took 1.462 ms (blocks of 32, 3.23 times faster) and 1.171 ms (blocks of 128, 4.04 times faster); bfloat16 weights in PyTorch halved the bytes and were 1.95 times faster. Unpacking 4-bit weights costs work, so the kernel matters. At 16 tokens the int4 speed-ups shrank to 1.57 and 2.29, because the work moves towards the compute-bound side. There bfloat16 was 2.80 times faster, more than its byte ratio, probably because this server CPU has bfloat16 matrix instructions that many phone CPUs, especially low-cost ones, lack. The int4 output error was 0.085 to 0.10 of the output's size, the price of 4 bits.
7. **What transfers to a phone.** Sizes, bits per weight and errors transfer exactly. Speed-ups do not: they depend on the kernel (here ONNX Runtime's) and on the chip's integer instructions, so measure them on the phone with the runtime you will ship.